# 13. 技術アセスメント（Technology Assessment） — 練習問題

**対象技術**: AI（生成AI・機械学習・自動化を含む）

技術アセスメントは、技術の便益・リスク・分配影響を「影響領域×ステークホルダー」のマトリクスで体系的に評価する上位フレームである。各セルは便益−リスクのネット値で、影響領域の重みで加重集計し、ステークホルダー間の便益分配の不平等をジニ係数で測る。さらに ELSI（倫理的・法的・社会的含意）チェックリストをスコアリングし、重点対応領域を特定する。このノートブックでは生成AIの社会実装を題材にする。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# --- 日本語フォント設定: 共通モジュール jp_font.py を読み込む ---
# フォント探索・登録・フォールバックの実装は repo 直下の jp_font.py に集約。
import os as _os, sys as _sys
_d = _os.path.abspath(_os.getcwd())
while not _os.path.exists(_os.path.join(_d, "jp_font.py")) and _d != _os.path.dirname(_d):
    _d = _os.path.dirname(_d)
_sys.path.insert(0, _d)
from jp_font import setup_japanese_font
setup_japanese_font()


## 影響領域・ステークホルダー・ネットインパクト行列の定義

8つの影響領域と7つのステークホルダーを定義し、各セルに便益−リスクのネット値（-5〜+5）を持つ行列を構築する。ELSI チェックリストもあわせて定義する。

In [ ]:
DOMAINS = ["雇用・労働", "教育・スキル", "公平性・バイアス", "偽情報・民主主義",
           "知的財産・著作権", "プライバシー", "環境(電力消費)", "安全保障"]
# 影響領域の重み(評価者の価値判断)。合計1.0。
DOMAIN_WEIGHTS = np.array([0.18, 0.12, 0.15, 0.15, 0.10, 0.10, 0.10, 0.10])

STAKEHOLDERS = [
    "労働者",
    "企業",
    "教育機関",
    "一般市民",
    "クリエイター",
    "政府",
    "グローバルサウス",
]

# ネットインパクト行列 (影響領域 行 × ステークホルダー 列)。
# 各セル = 便益 − リスク のネット値。-5〜+5。
NET_IMPACT = np.array([
    # 労働者  企業  教育機関 一般市民 クリエイタ 政府  Gサウス
    [  -3,    5,     1,     0,      -2,     2,    -2 ],   # 雇用・労働
    [   2,    3,     2,     3,       1,     1,     1 ],   # 教育・スキル
    [  -2,    1,    -1,    -2,      -1,    -1,    -4 ],   # 公平性・バイアス
    [  -1,    1,     0,    -4,      -1,    -2,    -3 ],   # 偽情報・民主主義
    [   0,    2,     0,     0,      -5,     0,    -1 ],   # 知的財産・著作権
    [  -1,    1,    -1,    -3,      -1,    -1,    -2 ],   # プライバシー
    [   0,   -1,     0,    -1,      -1,    -1,    -2 ],   # 環境(電力消費)
    [   0,    1,     0,    -2,       0,     1,    -3 ],   # 安全保障
])

# ELSI チェックリスト。各項目を 0(未対応)〜5(十分対応)でスコアリング。
ELSI_CHECKLIST = [
    ("学習データの著作権・出所に関する同意の確保",          1),
    ("アルゴリズムのバイアス・差別の監査と是正",            2),
    ("生成物の真偽判別(来歴・電子透かし)の整備",            2),
    ("自動化で職を失う労働者の再教育・移行支援",            1),
    ("訓練・推論の電力消費と環境負荷の開示",                3),
    ("AIの誇大宣伝・能力誤認に対する説明責任",              2),
]

print("[影響領域 × ステークホルダー ネットインパクト行列] (便益−リスク)")
print("-" * 78)
header = "  領域\\主体".ljust(18) + "".join(f"{s[:6]:>9}" for s in STAKEHOLDERS)
print(header)
for d, dom in enumerate(DOMAINS):
    row = f"  {dom:<14}" + "".join(f"{v:>9d}" for v in NET_IMPACT[d])
    print(row)

## 1. 影響領域別の合算と重み

影響領域別の全主体合算ネット値を確認したうえで、評価者の価値判断である影響領域の重みを示す。

In [ ]:
print("[影響領域の重み]")
for dom, w in zip(DOMAINS, DOMAIN_WEIGHTS):
    print(f"  {dom:<14} {w:.2f}")
domain_total = NET_IMPACT.sum(axis=1)
print("\n[影響領域別 全主体合算ネット値]")
for d, dom in enumerate(DOMAINS):
    print(f"  {dom:<14} 合算={domain_total[d]:+d}")

## 2. ステークホルダー別の総合ネット便益

影響領域の重みで加重し、ステークホルダーごとの総合ネット便益を算出する。

In [ ]:
def weighted_net_benefit(net_impact, weights):
    """影響領域の重みで加重し、ステークホルダー別の総合ネット便益を返す。"""
    return weights @ net_impact


sh_benefit = weighted_net_benefit(NET_IMPACT, DOMAIN_WEIGHTS)
print("[ステークホルダー別 総合ネット便益] (影響領域重みで加重)")
print("-" * 70)
order = np.argsort(-sh_benefit)
for si in order:
    bar_len = int(abs(sh_benefit[si]) * 6)
    bar = ("+" if sh_benefit[si] >= 0 else "-") * max(bar_len, 1)
    print(f"  {STAKEHOLDERS[si]:<18} {sh_benefit[si]:>7.3f}  {bar}")

## 3. 便益分配の不平等評価（ジニ係数）

ステークホルダー間のネット便益の散らばりをジニ係数で測る。便益にマイナスが含まれるため、最小値0シフトで非負化してから計算する。

In [ ]:
def gini_coefficient(values):
    """値の不平等を測るジニ係数。負値を含むため最小値0シフトで前処理する。
    0に近いほど平等、1に近いほど一部に集中。
    """
    v = np.asarray(values, dtype=float)
    v = v - v.min()                 # 非負化(最小値を0に)
    if v.sum() == 0:
        return 0.0
    v = np.sort(v)
    n = len(v)
    index = np.arange(1, n + 1)
    return float(np.sum((2 * index - n - 1) * v) / (n * np.sum(v)))


gini = gini_coefficient(sh_benefit)
print(f"[便益分配の不平等] ジニ係数(非負化後) = {gini:.3f}")
worst = STAKEHOLDERS[int(np.argmin(sh_benefit))]
best = STAKEHOLDERS[int(np.argmax(sh_benefit))]
print(f"  最大便益: {best} / 最小便益: {worst}")
print(f"  -> 便益は一部主体に偏在。総便益が正でも『{worst}』はリスクのみを")
print("     負う構図であり、TAはこの偏りを政策オプションへ折り返すべき。")

## 4. ELSI チェックリストの評価

ELSI チェックリストの各項目をスコアリングし、低スコア（≦2）の重点対応領域を特定する。

In [ ]:
print("[ELSI チェックリスト評価] (0:未対応 〜 5:十分対応)")
print("-" * 70)
scores = []
for item, score in ELSI_CHECKLIST:
    scores.append(score)
    bar = "#" * score + "." * (5 - score)
    flag = "  <= 重点対応" if score <= 2 else ""
    print(f"  [{bar}] {item}{flag}")
scores = np.array(scores)
print("-" * 70)
print(f"  ELSI平均スコア = {scores.mean():.2f} / 5.00")
low_items = [item for item, s in ELSI_CHECKLIST if s <= 2]
print(f"  重点対応すべき低スコア項目: {len(low_items)} 件")

## 可視化1: 影響領域×ステークホルダーのインパクト行列ヒートマップ

ネットインパクト行列を発散カラーマップで描く。便益（＋）は青、リスク（−）は赤で表示し、便益とリスクがどの領域・主体に偏在するかを可視化する。

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.5))
im = ax.imshow(NET_IMPACT, cmap="RdBu", vmin=-5, vmax=5, aspect="auto")
dom_labels = ["Jobs", "Education", "Fairness", "Disinfo",
              "IP/Copyright", "Privacy", "Environment", "Security"]
sh_labels = ["Workers", "Firms", "Schools", "Citizens",
             "Creators", "Gov", "GlobalSouth"]
ax.set_xticks(range(len(STAKEHOLDERS)))
ax.set_xticklabels(sh_labels, rotation=20, ha="right")
ax.set_yticks(range(len(DOMAINS)))
ax.set_yticklabels(dom_labels)
for d in range(len(DOMAINS)):
    for s in range(len(STAKEHOLDERS)):
        ax.text(s, d, f"{NET_IMPACT[d, s]:+d}", ha="center", va="center",
                fontsize=9, color="#222222")
ax.set_title("Net impact (benefit - risk): Domain x Stakeholder")
fig.colorbar(im, ax=ax, label="net value  (- risk / + benefit)")
fig.tight_layout()
plt.show()

## 可視化2: 便益分配のローレンツ曲線とジニ係数

ステークホルダー別ネット便益（非負化後）のローレンツ曲線を描き、完全平等線からの乖離としてジニ係数を図示する。曲線が対角線から離れるほど便益が一部に集中している。

In [ ]:
v = np.sort(sh_benefit - sh_benefit.min())  # 非負化して昇順
cum = np.concatenate([[0], np.cumsum(v)])
cum = cum / cum[-1]
xs = np.linspace(0, 1, len(cum))

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], color="#999999", linestyle="--",
        label="line of equality")
ax.plot(xs, cum, color="#1f3f5c", linewidth=2, marker="o",
        label="Lorenz curve")
ax.fill_between(xs, cum, xs, color="#7fb8d6", alpha=0.4)
ax.set_xlabel("Cumulative share of stakeholders")
ax.set_ylabel("Cumulative share of net benefit")
ax.set_title(f"Lorenz curve of net benefit  (Gini = {gini:.3f})")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend(loc="upper left")
fig.tight_layout()
plt.show()

## 未来デザイン論文での使われ方と結論への影響

未来デザイン論文において技術アセスメントは、ある技術の影響を単一の指標に還元せず、複数の影響領域と複数のステークホルダーにまたがって体系的に棚卸しするために用いられる。論文は影響領域×ステークホルダーのマトリクスを組み、便益・リスク・それらの分配を評価し、最後にガバナンス上の政策オプションを提示する。そこで生まれる結論は、多領域・多主体のバランスシートと、それに基づく政策提言という型をとる。「便益はおおむねA層に集まり、リスクはB層に偏って降りかかる、ゆえに政策Pが要請される」という、分配を明示した統治志向の言明が成果物となる。

この手法が結論に持ち込む特徴は、価値判断を隠さず前面に置くことである。包括性と分配の公平性そのものが評価軸として組み込まれ、論文は中立な記述にとどまらず規範的な提言へと踏み出す。これは技術アセスメントの長所だが、同時に「どの影響領域を立て、各領域にどの重みを与えるか」という設定に評価者の価値観が直接流れ込むことを意味する。重みづけ次第で、同じ技術が便益超過とも害悪超過とも結論されうる。

境界設定の面では、マトリクスに立てた影響領域とステークホルダーの一覧が、そのまま結論の射程を画定する。声を上げにくい主体や、将来世代のように代理人を持たない利害が一覧から漏れれば、その不利益はバランスシートに計上されず、分配の評価から構造的に抜け落ちる。時間観としては、未来を制度設計によって舵取りできる対象として扱い、その点で能動的だが、評価時点で見えている影響領域に縛られるため、想定外の長期的影響を取りこぼしやすい。したがってこの手法を用いた論文の説得力は、影響領域の重みを揺らす感度分析と、誰がマトリクスから漏れているかへの自覚に支えられる。

## 発展課題

**課題A**: 影響領域の重み `DOMAIN_WEIGHTS` を変えて感度分析せよ。例えば偽情報・民主主義重視/雇用重視の重み配分を用意し、総合評価とジニ係数がどう変わるかを比較し、価値判断が結論を左右する度合いを論ぜよ。

**課題B**: 政策オプション（再教育支援/著作権ライセンス制度/来歴表示の義務化など）を、特定ステークホルダーのネット便益に加点する効果としてモデル化し、分配公平性（ジニ係数の低下）の観点から政策オプションを比較せよ。